# Notebook 06 — Robustness Evaluation
## Adaptive Reliability-Aware Fusion (ARAF) Project

**Goal of this notebook:**
Systematically evaluate all four trained models under every corruption type
and severity level. This produces the main results figures for your paper.

**What makes this the most important notebook:**
Everything we built — the corruption module, the reliability estimator,
the adaptive fusion — was designed to shine here. Clean accuracy (Notebook 05)
was always expected to be similar across models. Corrupted accuracy is where
ARAF should pull ahead.

**Figures produced in this notebook:**
1. Robustness curves — accuracy vs severity for each corruption type
2. Corruption heatmap — accuracy table across all corruption types and severities
3. Trained reliability heatmap — what the estimator learned (vs random in NB04)
4. Modality dropout analysis — how models behave when one modality is missing
5. Radar chart — overall robustness profile per model
6. Summary results table — the table that goes in your paper

---


## 1. Imports and setup


In [ ]:
import os, sys, json, copy, random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

import torch
import torch.nn.functional as F
from torchvision import transforms
from transformers import BertTokenizer
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224
MAX_TEXT_LEN  = 32

clean_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Setup complete.")


## 2. Reload all components and trained models

We import from the `.py` files and reload the trained models from
the current session. If you are starting a fresh kernel, the models
variable must be available from Notebook 05.


In [ ]:
@dataclass
class MultimodalSample:
    image: torch.Tensor
    text_ids: torch.Tensor
    attention_mask: torch.Tensor
    label: torch.Tensor
    raw_image: Optional[object] = None
    raw_text: str = ""
    dataset_name: str = "vqa_v2"
    sample_id: str = ""
    image_corrupted: bool = False
    text_corrupted: bool = False
    image_missing: bool = False
    text_missing: bool = False
    corruption_severity: float = 0.0

import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

baselines_mod  = load_module("baselines",  "models/baselines.py")
araf_mod       = load_module("araf",       "models/araf.py")
corr_mod       = load_module("corruption", "corruption/corruption_module.py")

vqa_accuracy       = baselines_mod.vqa_accuracy
vqa_loss           = baselines_mod.vqa_loss
UnimodalImageModel = baselines_mod.UnimodalImageModel
UnimodalTextModel  = baselines_mod.UnimodalTextModel
NaiveFusionModel   = baselines_mod.NaiveFusionModel
ARAFModel          = araf_mod.ARAFModel
CorruptionModule   = corr_mod.CorruptionModule

print("Modules loaded.")


In [ ]:
# ── Load tokenizer and data ───────────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
with open("answer_vocab.json") as f:
    answer2idx = json.load(f)
idx2answer  = {v: k for k, v in answer2idx.items()}
NUM_CLASSES = len(answer2idx)

print("Loading VQA v2 validation data...")
hf_val_full = load_dataset("lmms-lab/VQAv2", split="validation",
                           trust_remote_code=True)
split  = hf_val_full.train_test_split(test_size=0.2, seed=42)
hf_val = split["test"]

def load_sample(row):
    pil = row["image"].convert("RGB")
    img = clean_transform(pil)
    enc = tokenizer(row["question"], padding="max_length",
                    max_length=MAX_TEXT_LEN, truncation=True,
                    return_tensors="pt")
    label = torch.zeros(NUM_CLASSES)
    cnt = Counter(a["answer"].lower().strip() for a in row["answers"])
    for ans, c in cnt.items():
        if ans in answer2idx:
            label[answer2idx[ans]] = min(c / 3.0, 1.0)
    return MultimodalSample(
        image=img, text_ids=enc["input_ids"].squeeze(0),
        attention_mask=enc["attention_mask"].squeeze(0),
        label=label, raw_image=pil, raw_text=row["question"],
        dataset_name="vqa_v2",
        sample_id=str(row.get("question_id", 0)),
    )

# Load evaluation samples
N_EVAL = 200  # increase to 500+ for paper results
print(f"Loading {N_EVAL} evaluation samples...")
eval_samples = [load_sample(hf_val[i]) for i in range(N_EVAL)]
print(f"Loaded {len(eval_samples)} samples.")


In [ ]:
# ── Use trained models from Notebook 05 ──────────────────────────────────────
# If running in the same session as Notebook 05, use:
#   trained_models = models  (already in memory)
# If starting fresh, you need to rebuild and reload checkpoints.

try:
    # Same session — models already trained
    trained_models = models
    print("Using trained models from current session.")
    for name, model in trained_models.items():
        model.eval()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  {name}: {trainable:,} trainable params")
except NameError:
    # Fresh session — rebuild models
    print("Models not in memory. Rebuilding architecture...")
    print("NOTE: Checkpoints saved frozen encoder weights separately.")
    print("Rebuilding with pretrained encoders + random classifier heads.")
    print("For best results, run Notebook 05 first in the same session.")

    NUM_CLASSES_LOCAL = NUM_CLASSES
    trained_models = {
        "UnimodalImage" : UnimodalImageModel(num_classes=NUM_CLASSES_LOCAL).to(DEVICE),
        "UnimodalText"  : UnimodalTextModel(num_classes=NUM_CLASSES_LOCAL).to(DEVICE),
        "NaiveFusion"   : NaiveFusionModel(num_classes=NUM_CLASSES_LOCAL).to(DEVICE),
        "ARAF"          : ARAFModel(num_classes=NUM_CLASSES_LOCAL).to(DEVICE),
    }
    for m in trained_models.values():
        m.eval()
    print("Models rebuilt (untrained classifiers).")


## 3. Evaluation infrastructure

We build a single evaluation function that:
1. Takes a model, a list of samples, and a corruption config
2. Applies the corruption to each sample
3. Runs the model and computes VQA accuracy
4. Returns the mean accuracy

This function is called hundreds of times across different corruption
types and severity levels, so it needs to be fast and correct.


In [ ]:
def evaluate(model, samples, corruption_module=None,
             model_name="", batch_size=32) -> float:
    """
    Evaluate a model on a list of samples, optionally with corruption.

    Args:
        model            : trained model
        samples          : list of MultimodalSample (clean)
        corruption_module: if not None, corrupt each sample before eval
        model_name       : used for ARAF-specific loss handling
        batch_size       : process this many samples at once

    Returns:
        mean VQA accuracy over all samples
    """
    model.eval()
    all_accs = []

    with torch.no_grad():
        for start in range(0, len(samples), batch_size):
            batch_samples = samples[start:start+batch_size]

            # Apply corruption if specified
            if corruption_module is not None:
                batch_samples = [corruption_module(s) for s in batch_samples]

            # Build batch dict
            batch = {
                "image"            : torch.stack([s.image for s in batch_samples]).to(DEVICE),
                "text_ids"         : torch.stack([s.text_ids for s in batch_samples]).to(DEVICE),
                "attention_mask"   : torch.stack([s.attention_mask for s in batch_samples]).to(DEVICE),
                "label"            : torch.stack([s.label for s in batch_samples]).to(DEVICE),
                "image_corrupted"  : [s.image_corrupted for s in batch_samples],
                "text_corrupted"   : [s.text_corrupted for s in batch_samples],
                "image_missing"    : [s.image_missing for s in batch_samples],
                "text_missing"     : [s.text_missing for s in batch_samples],
                "corruption_severity": [s.corruption_severity for s in batch_samples],
                "raw_text"         : [s.raw_text for s in batch_samples],
                "sample_id"        : [s.sample_id for s in batch_samples],
            }

            output = model(batch)
            acc    = vqa_accuracy(output["logits"], batch["label"])
            all_accs.append(acc)

    return float(np.mean(all_accs))


# ── Quick sanity check ────────────────────────────────────────────────────────
print("Sanity check — clean accuracy:")
for name, model in trained_models.items():
    acc = evaluate(model, eval_samples[:50], model_name=name)
    print(f"  {name:<20}: {acc:.4f}")
print()
print("These should match the val_acc from Notebook 05 training.")


## 4. Robustness curves — accuracy vs severity

This is the main result figure. We evaluate each model at each
corruption severity level (0=clean, 1-5=increasing corruption)
for each corruption type.

The key thing to look for:
- All models should perform similarly at severity 0 (clean)
- As severity increases, models should diverge
- ARAF should degrade more slowly than NaiveFusion
- UnimodalText should stay flat under image-only corruption
- UnimodalImage should collapse fast under image corruption

This pattern directly demonstrates ARAF's advantage.


In [ ]:
# Define all corruption configurations to evaluate
CORRUPTION_TYPES = {
    # Image corruptions
    "gaussian_noise" : {"is_image": True},
    "motion_blur"    : {"is_image": True},
    "occlusion"      : {"is_image": True},
    # Text corruptions
    "token_dropout"  : {"is_image": False},
    "token_shuffle"  : {"is_image": False},
    "token_mask"     : {"is_image": False},
}

SEVERITIES    = [0, 1, 2, 3, 4, 5]
MODEL_COLORS  = {
    "UnimodalImage": "#7F77DD",
    "UnimodalText" : "#1D9E75",
    "NaiveFusion"  : "#D85A30",
    "ARAF"         : "#185FA5",
}
MODEL_NAMES   = list(trained_models.keys())

def build_corruption_module(corruption_type, severity, is_image):
    """Build a deterministic corruption module for evaluation."""
    if severity == 0:
        return None  # clean
    return CorruptionModule(
        p_corrupt_image  = 1.0 if is_image else 0.0,
        p_corrupt_text   = 0.0 if is_image else 1.0,
        p_missing_image  = 0.0,
        p_missing_text   = 0.0,
        severity         = severity,
        image_corruptions= [corruption_type] if is_image else None,
        text_corruptions = [corruption_type] if not is_image else None,
    )

print("Running robustness evaluation...")
print(f"Evaluating {len(CORRUPTION_TYPES)} corruption types x "
      f"{len(SEVERITIES)} severities x {len(MODEL_NAMES)} models")
print(f"= {len(CORRUPTION_TYPES)*len(SEVERITIES)*len(MODEL_NAMES)} evaluations")
print(f"Using {N_EVAL} samples per evaluation.")
print()

# Results dict: results[corruption_type][model_name][severity] = accuracy
results = {ct: {mn: {} for mn in MODEL_NAMES}
           for ct in CORRUPTION_TYPES}

for ct, cfg in CORRUPTION_TYPES.items():
    print(f"  Corruption: {ct}")
    for sev in SEVERITIES:
        corr = build_corruption_module(ct, sev, cfg["is_image"])
        for name, model in trained_models.items():
            acc = evaluate(model, eval_samples, corr, name)
            results[ct][name][sev] = acc
        accs = [results[ct][n][sev] for n in MODEL_NAMES]
        print(f"    Severity {sev}: " +
              " | ".join(f"{n[:4]}={a:.3f}" for n, a in zip(MODEL_NAMES, accs)))

print()
print("Evaluation complete.")


In [ ]:
def plot_robustness_curves(results, corruption_types_to_plot=None):
    """
    Plot accuracy vs severity for each corruption type.
    One subplot per corruption type, all models overlaid.
    """
    if corruption_types_to_plot is None:
        corruption_types_to_plot = list(results.keys())

    n_plots = len(corruption_types_to_plot)
    n_cols  = 3
    n_rows  = (n_plots + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols*5, n_rows*4))
    axes = axes.flatten()

    for idx, ct in enumerate(corruption_types_to_plot):
        ax = axes[idx]
        for name in MODEL_NAMES:
            accs = [results[ct][name][s] for s in SEVERITIES]
            ax.plot(SEVERITIES, accs, marker="o", linewidth=2,
                    color=MODEL_COLORS[name], label=name, markersize=5)
            # Shade area under curve
            ax.fill_between(SEVERITIES, accs, alpha=0.07,
                            color=MODEL_COLORS[name])

        ax.set_title(ct.replace("_", " ").title(), fontsize=11)
        ax.set_xlabel("Severity (0=clean)")
        ax.set_ylabel("VQA Accuracy")
        ax.set_xticks(SEVERITIES)
        ax.grid(alpha=0.3)
        ax.set_ylim(0, max(
            results[ct][n][0] for n in MODEL_NAMES) * 1.2 + 0.01)

    # Shared legend
    handles = [plt.Line2D([0], [0], color=MODEL_COLORS[n],
                          linewidth=2, label=n) for n in MODEL_NAMES]
    fig.legend(handles=handles, loc="lower center", ncol=4,
               fontsize=10, bbox_to_anchor=(0.5, -0.02))

    # Hide unused axes
    for i in range(len(corruption_types_to_plot), len(axes)):
        axes[i].axis("off")

    fig.suptitle("Robustness curves — accuracy vs corruption severity",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig("robustness_curves_trained.png", dpi=120,
                bbox_inches="tight")
    plt.show()
    print("Saved: robustness_curves_trained.png")

plot_robustness_curves(results)


## 5. Corruption heatmap

A compact summary of all results in one figure.
Rows = corruption types, columns = severity levels, cells = accuracy.
One heatmap per model, arranged side by side.

This is a standard figure in robustness papers — it gives reviewers
a complete picture at a glance.


In [ ]:
def plot_corruption_heatmap(results):
    """
    Plot a heatmap of accuracy for each model across all
    corruption types and severity levels.
    """
    corruption_list = list(results.keys())
    severity_list   = SEVERITIES
    n_models        = len(MODEL_NAMES)

    fig, axes = plt.subplots(1, n_models,
                             figsize=(n_models * 5, len(corruption_list) * 0.8 + 1))

    # Custom colormap: red (low) -> yellow -> green (high)
    cmap = LinearSegmentedColormap.from_list(
        "rg", ["#E24B4A", "#FAC775", "#1D9E75"])

    # Find global min/max for consistent colorscale
    all_vals = [results[ct][mn][sev]
                for ct in corruption_list
                for mn in MODEL_NAMES
                for sev in severity_list]
    vmin, vmax = min(all_vals), max(all_vals)

    for ax, model_name in zip(axes, MODEL_NAMES):
        matrix = np.array([
            [results[ct][model_name][sev] for sev in severity_list]
            for ct in corruption_list
        ])

        im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax,
                       aspect="auto")

        # Axis labels
        ax.set_xticks(range(len(severity_list)))
        ax.set_xticklabels([str(s) for s in severity_list], fontsize=9)
        ax.set_xlabel("Severity", fontsize=9)

        if ax == axes[0]:
            ax.set_yticks(range(len(corruption_list)))
            ax.set_yticklabels(
                [c.replace("_", " ") for c in corruption_list], fontsize=8)
        else:
            ax.set_yticks([])

        ax.set_title(model_name, fontsize=10, fontweight="bold",
                     color=MODEL_COLORS[model_name])

        # Annotate cells
        for i in range(len(corruption_list)):
            for j in range(len(severity_list)):
                val = matrix[i, j]
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=7,
                        color="white" if val < (vmin+vmax)/2 else "black")

        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle("Accuracy heatmap across corruption types and severities",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig("corruption_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: corruption_heatmap.png")

plot_corruption_heatmap(results)


## 6. Missing modality analysis

This evaluates the extreme case: what happens when an entire modality
is completely absent? This is the most important test for ARAF because
the reliability estimator should output near-zero for the missing modality
and route all information through the surviving one.


In [ ]:
def evaluate_missing_modality(models_dict, samples):
    """
    Evaluate all models under four missing modality scenarios.
    Returns dict of results.
    """
    scenarios = {
        "Clean"         : CorruptionModule(p_corrupt_image=0.0,
                            p_corrupt_text=0.0,
                            p_missing_image=0.0, p_missing_text=0.0),
        "Image missing" : CorruptionModule(p_corrupt_image=0.0,
                            p_corrupt_text=0.0,
                            p_missing_image=1.0, p_missing_text=0.0),
        "Text missing"  : CorruptionModule(p_corrupt_image=0.0,
                            p_corrupt_text=0.0,
                            p_missing_image=0.0, p_missing_text=1.0),
        "Both missing"  : CorruptionModule(p_corrupt_image=0.0,
                            p_corrupt_text=0.0,
                            p_missing_image=1.0, p_missing_text=1.0),
    }

    missing_results = {}
    print("Missing modality evaluation:")
    print(f"{'Scenario':<20} " +
          " ".join(f"{n[:12]:>12}" for n in MODEL_NAMES))
    print("-" * 70)

    for scenario_name, corr in scenarios.items():
        missing_results[scenario_name] = {}
        row_accs = []
        for name, model in models_dict.items():
            acc = evaluate(model, samples, corr, name)
            missing_results[scenario_name][name] = acc
            row_accs.append(acc)
        print(f"{scenario_name:<20} " +
              " ".join(f"{a:>12.4f}" for a in row_accs))

    return missing_results

missing_results = evaluate_missing_modality(trained_models, eval_samples)


In [ ]:
def plot_missing_modality(missing_results):
    """Bar chart comparing model performance under missing modality scenarios."""
    scenarios  = list(missing_results.keys())
    x          = np.arange(len(scenarios))
    bar_width  = 0.2

    fig, ax = plt.subplots(figsize=(12, 5))

    for i, name in enumerate(MODEL_NAMES):
        accs = [missing_results[s][name] for s in scenarios]
        bars = ax.bar(x + i * bar_width, accs, bar_width,
                      label=name, color=MODEL_COLORS[name], alpha=0.85)
        # Value labels on bars
        for bar, acc in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.002,
                    f"{acc:.3f}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x + bar_width * (len(MODEL_NAMES)-1) / 2)
    ax.set_xticklabels(scenarios, fontsize=10)
    ax.set_ylabel("VQA Accuracy", fontsize=11)
    ax.set_title("Model performance under missing modality scenarios",
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylim(0, max(
        missing_results[s][n]
        for s in scenarios for n in MODEL_NAMES) * 1.25 + 0.01)

    plt.tight_layout()
    plt.savefig("missing_modality_results.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: missing_modality_results.png")
    print()
    print("Key things to look for:")
    print("  UnimodalImage under 'Image missing'  -> should collapse to ~0")
    print("  UnimodalText  under 'Text missing'   -> should collapse to ~0")
    print("  ARAF vs NaiveFusion under both missing scenarios -> ARAF more graceful")

plot_missing_modality(missing_results)


## 7. Trained reliability heatmap

This is the figure that directly shows what ARAF learned.
Compare this to the random-weights version from Notebook 04.

After training:
- Clean inputs should get high reliability scores (green)
- Missing modalities should get near-zero scores (red)
- Corrupted inputs should get intermediate scores
- Image corruption should affect image scores but NOT text scores
- Text corruption should affect text scores but NOT image scores

This cross-modal independence is the key qualitative finding.


In [ ]:
def visualize_trained_reliability(araf_model, base_samples, n=8):
    """
    Show reliability scores for the TRAINED ARAF model.
    This should look very different from the random-weights version in NB04.
    """
    araf_model.eval()

    scenarios = {
        "Clean"        : CorruptionModule(p_corrupt_image=0.0,
                           p_corrupt_text=0.0,
                           p_missing_image=0.0, p_missing_text=0.0),
        "ImgNoise s1"  : CorruptionModule(p_corrupt_image=1.0,
                           p_corrupt_text=0.0, p_missing_image=0.0,
                           p_missing_text=0.0, severity=1,
                           image_corruptions=["gaussian_noise"]),
        "ImgNoise s3"  : CorruptionModule(p_corrupt_image=1.0,
                           p_corrupt_text=0.0, p_missing_image=0.0,
                           p_missing_text=0.0, severity=3,
                           image_corruptions=["gaussian_noise"]),
        "ImgNoise s5"  : CorruptionModule(p_corrupt_image=1.0,
                           p_corrupt_text=0.0, p_missing_image=0.0,
                           p_missing_text=0.0, severity=5,
                           image_corruptions=["gaussian_noise"]),
        "Img missing"  : CorruptionModule(p_corrupt_image=0.0,
                           p_corrupt_text=0.0,
                           p_missing_image=1.0, p_missing_text=0.0),
        "TxtDrop s3"   : CorruptionModule(p_corrupt_image=0.0,
                           p_corrupt_text=1.0, p_missing_image=0.0,
                           p_missing_text=0.0, severity=3,
                           text_corruptions=["token_dropout"]),
        "Txt missing"  : CorruptionModule(p_corrupt_image=0.0,
                           p_corrupt_text=0.0,
                           p_missing_image=0.0, p_missing_text=1.0),
    }

    scenario_names = list(scenarios.keys())
    n_scenarios    = len(scenario_names)
    n_samples      = min(n, len(base_samples))

    img_scores = np.zeros((n_samples, n_scenarios))
    txt_scores = np.zeros((n_samples, n_scenarios))

    def make_batch(s):
        return {
            "image"            : s.image.unsqueeze(0).to(DEVICE),
            "text_ids"         : s.text_ids.unsqueeze(0).to(DEVICE),
            "attention_mask"   : s.attention_mask.unsqueeze(0).to(DEVICE),
            "label"            : s.label.unsqueeze(0).to(DEVICE),
            "image_corrupted"  : [s.image_corrupted],
            "text_corrupted"   : [s.text_corrupted],
            "image_missing"    : [s.image_missing],
            "text_missing"     : [s.text_missing],
            "corruption_severity": [s.corruption_severity],
            "raw_text"         : [s.raw_text],
            "sample_id"        : [s.sample_id],
        }

    with torch.no_grad():
        for j, (sname, corr) in enumerate(scenarios.items()):
            for i in range(n_samples):
                s   = corr(base_samples[i])
                out = araf_model(make_batch(s))
                img_scores[i, j] = out["img_reliability"].item()
                txt_scores[i, j] = out["txt_reliability"].item()

    cmap = LinearSegmentedColormap.from_list(
        "rg", ["#E24B4A", "#FAC775", "#1D9E75"])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for ax, scores, title in zip(
        axes,
        [img_scores, txt_scores],
        ["Image reliability (trained)", "Text reliability (trained)"]
    ):
        im = ax.imshow(scores, vmin=0, vmax=1, cmap=cmap, aspect="auto")
        ax.set_xticks(range(n_scenarios))
        ax.set_xticklabels(scenario_names, rotation=30, ha="right", fontsize=8)
        ax.set_yticks(range(n_samples))
        ax.set_yticklabels([f"S{i}" for i in range(n_samples)], fontsize=8)
        ax.set_title(title, fontsize=11)
        for i in range(n_samples):
            for j in range(n_scenarios):
                ax.text(j, i, f"{scores[i,j]:.2f}",
                        ha="center", va="center", fontsize=7,
                        color="white" if scores[i,j] < 0.5 else "black")
        plt.colorbar(im, ax=ax, label="Reliability")

    fig.suptitle(
        "ARAF reliability scores after training" + "
"
        "Compare to Notebook 04 random-weights version",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig("reliability_scores_trained.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: reliability_scores_trained.png")

visualize_trained_reliability(trained_models["ARAF"], eval_samples, n=8)


## 8. Radar chart — overall robustness profile

A radar chart shows each model's robustness across all corruption
types simultaneously. Each axis is one corruption type, and the
value is the mean accuracy across severity levels 1-5 (corrupted only).

A model with a larger area is more robust overall.
This is a great figure for presentations and papers.


In [ ]:
def plot_radar_chart(results):
    """
    Radar chart showing mean corrupted accuracy per corruption type per model.
    Each axis = one corruption type.
    Value = mean accuracy across severity levels 1-5.
    """
    corruption_list = list(results.keys())
    N = len(corruption_list)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=(8, 8),
                           subplot_kw=dict(projection="polar"))

    for name in MODEL_NAMES:
        # Mean accuracy across severity 1-5 (not 0) per corruption type
        values = [np.mean([results[ct][name][s]
                           for s in SEVERITIES if s > 0])
                  for ct in corruption_list]
        values += values[:1]  # close polygon

        ax.plot(angles, values, linewidth=2,
                color=MODEL_COLORS[name], label=name)
        ax.fill(angles, values, alpha=0.1, color=MODEL_COLORS[name])

    # Axis labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [c.replace("_", "
") for c in corruption_list], fontsize=9)

    ax.set_title("Robustness profile" + "
" +
                 "(larger area = more robust)",
                 fontsize=12, pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("radar_chart.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: radar_chart.png")

plot_radar_chart(results)


## 9. Summary results table

The clean, paper-ready results table. This is what goes in your
paper's experimental results section.

Rows = models, columns = clean + each corruption type (mean across severities).
The last column is the overall mean corrupted accuracy — your key metric.


In [ ]:
def print_results_table(results, missing_results):
    """Print a clean results table suitable for a paper."""
    corruption_list = list(results.keys())

    # Header
    header = f"{'Model':<20} {'Clean':>8}"
    for ct in corruption_list:
        header += f" {ct[:8]:>10}"
    header += f" {'Mean Corr':>10}"
    print("=" * len(header))
    print("RESULTS TABLE")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    summary = {}
    for name in MODEL_NAMES:
        clean_acc  = results[corruption_list[0]][name][0]
        corr_accs  = []
        row = f"{name:<20} {clean_acc:>8.4f}"
        for ct in corruption_list:
            mean_corr = np.mean([results[ct][name][s]
                                 for s in SEVERITIES if s > 0])
            corr_accs.append(mean_corr)
            row += f" {mean_corr:>10.4f}"
        mean_all = np.mean(corr_accs)
        row += f" {mean_all:>10.4f}"
        print(row)
        summary[name] = {
            "clean"    : clean_acc,
            "mean_corr": mean_all,
            "per_corr" : dict(zip(corruption_list, corr_accs)),
        }

    print("=" * len(header))
    print()

    # ARAF vs NaiveFusion improvement
    araf_mean  = summary["ARAF"]["mean_corr"]
    naive_mean = summary["NaiveFusion"]["mean_corr"]
    improvement = (araf_mean - naive_mean) / naive_mean * 100

    print("Key findings:")
    print(f"  ARAF mean corrupted accuracy    : {araf_mean:.4f}")
    print(f"  NaiveFusion mean corrupted acc  : {naive_mean:.4f}")
    print(f"  ARAF improvement over Naive     : {improvement:+.1f}%")
    print()

    # Robustness drop (clean - corrupted)
    print("Robustness drop (clean - mean_corrupted):")
    for name in MODEL_NAMES:
        drop = summary[name]["clean"] - summary[name]["mean_corr"]
        print(f"  {name:<20}: {drop:+.4f} "
              f"({'more robust' if drop < summary['NaiveFusion']['clean'] - naive_mean else 'less robust'} than NaiveFusion)")

    return summary

summary = print_results_table(results, missing_results)


## 10. Save all results


In [ ]:
# Save full results dict
eval_results = {
    "robustness"      : {ct: {mn: {str(s): v for s, v in sv.items()}
                              for mn, sv in mdict.items()}
                         for ct, mdict in results.items()},
    "missing_modality": {sc: {mn: float(v) for mn, v in mdict.items()}
                         for sc, mdict in missing_results.items()},
    "summary"         : {mn: {k: float(v) if isinstance(v, float) else
                               {k2: float(v2) for k2, v2 in v.items()}
                               for k, v in s.items()}
                         for mn, s in summary.items()},
    "n_eval_samples"  : N_EVAL,
}

with open("evaluation_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print("Saved: evaluation_results.json")
print()
print("All figures saved:")
figures = [
    "robustness_curves_trained.png",
    "corruption_heatmap.png",
    "missing_modality_results.png",
    "reliability_scores_trained.png",
    "radar_chart.png",
]
for fig in figures:
    exists = os.path.exists(fig)
    print(f"  {'OK' if exists else 'MISSING'} {fig}")


## 11. Project summary and next steps

### What you have built

A complete corruption-aware multimodal learning pipeline:

| Component | File | Status |
|---|---|---|
| Data contract | `MultimodalSample` dataclass | Done |
| Dataset loader | `VQAv2Dataset` | Done |
| Corruption module | `corruption/corruption_module.py` | Done |
| Image encoder | `models/baselines.py` | Done |
| Text encoder | `models/baselines.py` | Done |
| Baseline models | `models/baselines.py` | Done |
| ARAF model | `models/araf.py` | Done |
| Training loop | Notebook 05 | Done |
| Evaluation | Notebook 06 | Done |

### Figures for your paper / PPT

| Figure | File | Use |
|---|---|---|
| Architecture comparison | `architecture_comparison.png` | Introduction slide |
| Training curves | `training_curves.png` | Experiments section |
| Robustness curves | `robustness_curves_trained.png` | Main result |
| Corruption heatmap | `corruption_heatmap.png` | Supplementary |
| Missing modality | `missing_modality_results.png` | Ablation |
| Reliability heatmap | `reliability_scores_trained.png` | Qualitative analysis |
| Radar chart | `radar_chart.png` | Summary slide |

### How to scale up for publication-quality results

1. Set `MAX_TRAIN_SAMPLES = None` in Notebook 05 (full 200k samples)
2. Set `NUM_EPOCHS = 20` minimum
3. Set `N_EVAL = 1000` here for more reliable evaluation numbers
4. Consider unfreezing last 2 layers of encoders with LR=1e-5
5. Add Hateful Memes as a second dataset (only the loader changes)

### The research contribution in one sentence

ARAF improves robustness to modality corruption by estimating
per-modality reliability and weighting the fusion accordingly,
with negligible parameter overhead over naive concatenation fusion.
